# 🚀 Edge-Inference-Project: Google Colab Entry Point

This notebook serves as the master orchestrator for running the ML compression pipeline and inference server on Google Colab.

### 🏗️ Architecture Overview
1.  **Pruning**: Reduces model size by removing redundant channels.
2.  **Distillation**: Trains a tiny Student model to mimic a large Teacher.
3.  **Quantization**: Converts the model to INT8 ONNX for edge performance.
4.  **C++ Server**: A high-performance inference engine built with ONNX Runtime.

## 🛠️ Step 1: Environment Setup
Clone the repo and install dependencies.

In [ ]:
# @title 1. Clone & Install
import os

REPO_URL = "https://github.com/JakkalaNagaVamsiKrishna/edge-inference-project.git"
REPO_NAME = "edge-inference-project"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

print("Installing dependencies (this may take a minute)...")
!pip install -r requirements.txt --quiet
!pip install onnxruntime-gpu --quiet

print("✅ Setup Complete")

## 🧪 Step 2: Run Compression Pipeline
This executes Pruning, Distillation, and Quantization.

In [ ]:
# @title 2. Start ML Pipeline
!python3 -m compressor.pipeline

## ⚡ Step 3: Build & Run C++ Inference Server
We compile the high-performance C++ code and start it in the background.

In [ ]:
# @title 3. Compile & Start Server
!sudo apt-get update && sudo apt-get install -y libonnxruntime-dev --quiet

print("Compiling C++ Inference Server...")
!mkdir -p build && cd build && cmake .. && make -j$(nproc)

import subprocess
import time

print("Starting Server in background...")
server_proc = subprocess.Popen(["./build/inference_server"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Wait for model mapping
print("🚀 Server is Live at http://localhost:8080")

## 📊 Step 4: Benchmark & Dashboard
Run Hardware-in-the-Loop tests and view the results.

In [ ]:
# @title 4. Run Benchmark Runner
!python3 -m benchmark.runner --n 100

In [ ]:
# @title 5. Launch Observability Dashboard
from google.colab.output import serve_kernel_port

# Start Flask app in background
subprocess.Popen(["python3", "-m", "dashboard.app"])
time.sleep(2)

print("Click the link below to open the dashboard:")
serve_kernel_port(5000)